In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive/fast_code_project/

decode.cu  proj_conv_silu_kernel.cu


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

True
NVIDIA A100-SXM4-40GB


In [4]:
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.7 MB/s eta 0:00:00


## 0. Imports & Global Configuration

In [5]:
import math
import time
import warnings
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

warnings.filterwarnings('ignore')

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE  = torch.bfloat16          # BF16 matches A100 Tensor Core native format

if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {props.name}")
    print(f"SM count     : {props.multi_processor_count}")
    print(f"VRAM         : {props.total_memory / 1e9:.1f} GB")
    print(f"Compute cap  : {props.major}.{props.minor}")
else:
    print("WARNING: CUDA not available — running on CPU (benchmarks will be slow)")

# ── Architecture constants (Table I) ────────────────────────────────────────
H   = 16      # Number of attention heads
DK  = 128     # Per-head key/query dimension
DV  = 256     # Per-head value dimension
TDK = H * DK  # 2048 — Total key/query projection dim
TDV = H * DV  # 4096 — Total value projection dim
D   = 2048    # Model hidden size

# Benchmark sweep configurations
T_VALS = [1, 2048, 4096, 8192]   # T=1 → decode; larger → prefill
B_VALS = [1, 8, 32, 64]

WARMUP_ITERS = 10
BENCH_ITERS  = 50

print(f"\nArchitecture: H={H}, dk={DK}, dv={DV}, Dk={TDK}, Dv={TDV}, D={D}")

GPU : NVIDIA A100-SXM4-40GB
SM count     : 108
VRAM         : 42.4 GB
Compute cap  : 8.0

Architecture: H=16, dk=128, dv=256, Dk=2048, Dv=4096, D=2048


## 1. Benchmarking Utilities

In [6]:
def benchmark_fn(fn, warmup=WARMUP_ITERS, iters=BENCH_ITERS):
    """
    Time `fn()` with CUDA events for accurate GPU measurement.
    Returns mean latency in milliseconds.
    """
    if DEVICE.type == 'cuda':
        # Warm-up
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()

        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)

        start_ev.record()
        for _ in range(iters):
            fn()
        end_ev.record()
        torch.cuda.synchronize()

        return start_ev.elapsed_time(end_ev) / iters   # ms
    else:
        # CPU fallback
        for _ in range(warmup):
            fn()
        t0 = time.perf_counter()
        for _ in range(iters):
            fn()
        return (time.perf_counter() - t0) / iters * 1e3


def throughput_gb_s(bytes_accessed: int, latency_ms: float) -> float:
    """Convert byte count + latency to memory throughput in GB/s."""
    return bytes_accessed / (latency_ms * 1e-3) / 1e9


def flops_to_tflops(flops: int, latency_ms: float) -> float:
    return flops / (latency_ms * 1e-3) / 1e12


def make_df(results):
    """Convert list-of-dicts to a nicely formatted DataFrame."""
    df = pd.DataFrame(results)
    for col in df.select_dtypes(float).columns:
        df[col] = df[col].map(lambda x: f"{x:.4f}")
    return df


def plot_heatmap(df_raw, value_col, title, fmt=".3f", cmap="viridis"):
    """Pivot B×T results into a heatmap."""
    pivot = df_raw.pivot(index='B', columns='T', values=value_col).astype(float)
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(c) for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(r) for r in pivot.index])
    ax.set_xlabel('Sequence Length T')
    ax.set_ylabel('Batch Size B')
    ax.set_title(title)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.values[i, j]
            ax.text(j, i, format(v, fmt), ha='center', va='center', fontsize=8,
                    color='white' if v < pivot.values.max() * 0.6 else 'black')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


print("Benchmark utilities defined.")

Benchmark utilities defined.


---
## Kernel 4 — Decode Recurrence (T = 1)

### Design
When T = 1 the kernel is memory-bandwidth-bound: the entire state
`S ∈ ℝ^{d_k × d_v}` per `(b, h)` must be streamed from HBM, updated in-place, and written back.
The design **fuses** gating, the Delta-Rule update, and the output projection into a single pass
so the state is read and written exactly once.

Each thread owns a column `S[:,j]` (register-streaming strategy), computing:
```
r_j  = g · kᵀ S[:,j]
δ_j  = β (v_j − r_j)
S[:,j] ← g S[:,j] + δ_j k
o_j   = scale · qᵀ S[:,j]
```

### Compile Customized Cuda Kernel

In [7]:
!rm -rf /root/.cache/torch_extensions
!rm -rf /root/.cache/torch_extensions/*

In [8]:
from torch.utils.cpp_extension import load

_KERNEL_PATH = "/content/drive/MyDrive/fast_code_project/decode.cu"

print("Compiling decode.cu …")
_gdn_ext = load(
    name="gdn_decode_ext",
    sources=[_KERNEL_PATH],
    extra_cuda_cflags=[
        "-O3",
        "--use_fast_math",
        "-arch=sm_80",  # A100
    ],
    verbose=True
)
print("✓ CUDA extension compiled successfully.")

Compiling decode.cu …
✓ CUDA extension compiled successfully.


In [9]:
# python wrapper for customize cuda kernel
def gdn_decode_cuda_kernel(q, k, v, a_scalar, b_scalar, A_log, dt_bias, scale: float, S):
    B  = q.shape[0]
    H  = q.shape[1] # Make sure H is captured
    dv = v.shape[-1]

    o_out = torch.empty(B, H, dv, device=q.device, dtype=torch.float32)
    S_out = torch.empty_like(S)

    _gdn_ext.forward(
        q.contiguous(),
        k.contiguous(),
        v.contiguous(),
        a_scalar.contiguous(),
        b_scalar.contiguous(),
        A_log.contiguous(),
        dt_bias.contiguous(),
        S.contiguous(),
        scale,
        o_out,
        S_out
    )
    return o_out, S_out

### Define Pytorch Baseline

In [10]:
def gdn_decode_baseline(q, k, v, a_scalar, b_scalar, A_log, dt_bias, scale, S):
    """
    Unfused decode step: each sub-operation is separate.

    q, k  : (B, H, dk)   BF16
    v     : (B, H, dv)   BF16
    a_scalar, b_scalar : (B, H)  BF16 — per-head gate inputs for this token
    A_log, dt_bias     : (H,)    FP32
    scale : scalar
    S     : (B, H, dk, dv) FP32  — in-place modified

    Returns: o (B, H, dv) FP32, updated S
    """
    # Gate computation (FP32)
    raw = a_scalar.float() + dt_bias.float()           # (B, H)
    g   = torch.exp(-torch.exp(A_log.float()) *
                    F.softplus(raw))                    # (B, H)
    beta = torch.sigmoid(b_scalar.float())             # (B, H)

    kf = k.float()   # (B, H, dk)
    vf = v.float()   # (B, H, dv)
    qf = q.float()   # (B, H, dk)

    # Gated decay  (separate kernel in unfused)
    S = S * g[:, :, None, None]

    # Read
    r = torch.einsum('bhk,bhkd->bhd', kf, S)          # (B, H, dv)

    # Delta
    delta = beta[:, :, None] * (vf - r)               # (B, H, dv)

    # Rank-1 update
    S = S + torch.einsum('bhk,bhd->bhkd', kf, delta)

    # Output projection
    o = scale * torch.einsum('bhk,bhkd->bhd', qf, S)  # (B, H, dv)

    return o, S

@torch.compile(fullgraph=True, dynamic=False)
def gdn_decode_fused(q, k, v, a_scalar, b_scalar, A_log, dt_bias, scale, S):
    """
    Fused decode step compiled by torch.compile / inductor.
    All five sub-operations are merged into a minimum number of kernels,
    and the state S is streamed through registers / L2 only once.
    """
    raw  = a_scalar.float() + dt_bias.float()
    g    = torch.exp(-torch.exp(A_log.float()) * F.softplus(raw))
    beta = torch.sigmoid(b_scalar.float())

    kf, vf, qf = k.float(), v.float(), q.float()

    S_decayed = S * g[:, :, None, None]
    r         = torch.einsum('bhk,bhkd->bhd', kf, S_decayed)
    delta     = beta[:, :, None] * (vf - r)
    S_new     = S_decayed + torch.einsum('bhk,bhd->bhkd', kf, delta)
    o         = scale * torch.einsum('bhk,bhkd->bhd', qf, S_new)

    return o, S_new

### Correctness Check

In [11]:
# ── Correctness check ────────────────────────────────────────────────────────
with torch.no_grad():
    _B = 4
    _q    = torch.randn(_B, H, DK, device=DEVICE, dtype=DTYPE)
    _k    = F.normalize(torch.randn(_B, H, DK, device=DEVICE, dtype=DTYPE), dim=-1)
    _v    = torch.randn(_B, H, DV, device=DEVICE, dtype=DTYPE)
    _a_s  = torch.randn(_B, H,     device=DEVICE, dtype=DTYPE) * 0.1
    _b_s  = torch.randn(_B, H,     device=DEVICE, dtype=DTYPE) * 0.1
    _Alog = torch.randn(H,         device=DEVICE, dtype=torch.float32) * 0.1
    _dtb  = torch.randn(H,         device=DEVICE, dtype=torch.float32) * 0.1
    _S    = torch.zeros(_B, H, DK, DV, device=DEVICE, dtype=torch.float32)
    _scl  = 1.0 / math.sqrt(DK)

    o_base,  S_base  = gdn_decode_baseline(_q, _k, _v, _a_s, _b_s, _Alog, _dtb, _scl, _S.clone())
    o_fused, S_fused = gdn_decode_fused(   _q, _k, _v, _a_s, _b_s, _Alog, _dtb, _scl, _S.clone())
    o_cuda, S_cuda = gdn_decode_cuda_kernel(   _q, _k, _v, _a_s, _b_s, _Alog, _dtb, _scl, _S.clone())

    diff_o = (o_base - o_cuda).abs().max().item()
    diff_S = (S_base - S_cuda).abs().max().item()
    print(f"Decode correctness: max |o| = {diff_o:.2e}, max |S| = {diff_S:.2e}  ✓" if diff_o < 1e-2 else f"MISMATCH o: {diff_o}")

Decode correctness: max |o| = 1.49e-07, max |S| = 8.94e-08  ✓


### Benchmark Comparison

In [18]:
# ── Benchmark sweep — only T=1 for decode ────────────────────────────────────
kernel4_results = []

# A100 peak memory bandwidth ≈ 2 TB/s
A100_BW_TBS = 1.555
A100_PEAK_FP32_TFLOPS = 19.5

for B in B_VALS:
    T = 1  # Decode always T=1
    q    = torch.randn(B, H, DK, device=DEVICE, dtype=DTYPE)
    k    = F.normalize(torch.randn(B, H, DK, device=DEVICE, dtype=DTYPE), dim=-1)
    v    = torch.randn(B, H, DV, device=DEVICE, dtype=DTYPE)
    a_s  = torch.randn(B, H,     device=DEVICE, dtype=DTYPE) * 0.1
    b_s  = torch.randn(B, H,     device=DEVICE, dtype=DTYPE) * 0.1
    Alog = torch.randn(H,        device=DEVICE, dtype=torch.float32) * 0.1
    dtb  = torch.randn(H,        device=DEVICE, dtype=torch.float32) * 0.1
    scl  = 1.0 / math.sqrt(DK)

    S_base  = torch.zeros(B, H, DK, DV, device=DEVICE, dtype=torch.float32)
    S_fused = S_base.clone()

    def run_base():
        with torch.no_grad():
            gdn_decode_baseline(q, k, v, a_s, b_s, Alog, dtb, scl, S_base)

    def run_fused():
        with torch.no_grad():
            gdn_decode_fused(q, k, v, a_s, b_s, Alog, dtb, scl, S_fused)

    def run_ours():
        gdn_decode_cuda_kernel(q, k, v, a_s, b_s, Alog, dtb, scl, S_fused)

    lat_base  = benchmark_fn(run_base)
    lat_fused = benchmark_fn(run_fused)
    lat_cuda = benchmark_fn(run_ours)

    # Memory: read + write state (B, H, dk, dv) in FP32 = 4 bytes each
    state_bytes = B * H * DK * DV * 4 * 2   # read + write
    vec_bytes   = B * H * (DK + DK + DV) * 2   # q, k, v in BF16
    total_bytes = (B * H * DK * DV * 4 * 2) + (B * H * (DK + DK + DV) * 2)
    bw_cuda = throughput_gb_s(total_bytes, lat_cuda)
    hw_util = (bw_cuda / A100_BW_TBS * 1e3) * 100
    flops = 2 * B * H * DK * DV
    tflops_cuda = flops / (lat_cuda * 1e-3) / 1e12
    util_compute = (tflops_cuda / A100_PEAK_FP32_TFLOPS) * 100

    kernel4_results.append({
        "B": B,
        "Baseline (ms)": round(lat_base, 4),
        "Fused (ms)": round(lat_fused, 4),
        "CUDA (ms)": round(lat_cuda, 4),
        "Total Speedup": round(lat_base / lat_cuda, 2),
        "Kernel Speedup": round(lat_fused / lat_cuda, 2), # Gain over optimized PT
        "BW (GB/s)": round(bw_cuda, 1),
        "Util %": round(hw_util * 1e-6, 1)
    })

    print(f"B={B:>2} | Base: {lat_base:.4f} | Fused: {lat_fused:.4f} | CUDA: {lat_cuda:.4f} | Speedup: {lat_base/lat_cuda:.1f}x | TFLOPs: {round(tflops_cuda, 2)} | Compute Efficiency: {util_compute:.1f}%")

# Create and display the summary table
df_k4 = pd.DataFrame(kernel4_results)
print("\nKernel 1 results:")
df_k4

B= 1 | Base: 0.4825 | Fused: 0.3334 | CUDA: 0.0168 | Speedup: 28.8x | TFLOPs: 0.06 | Compute Efficiency: 0.3%
B= 8 | Base: 0.4522 | Fused: 0.3607 | CUDA: 0.0392 | Speedup: 11.5x | TFLOPs: 0.21 | Compute Efficiency: 1.1%
B=32 | Base: 0.5383 | Fused: 0.3220 | CUDA: 0.1612 | Speedup: 3.3x | TFLOPs: 0.21 | Compute Efficiency: 1.1%
B=64 | Base: 0.9624 | Fused: 0.6060 | CUDA: 0.3130 | Speedup: 3.1x | TFLOPs: 0.21 | Compute Efficiency: 1.1%

Kernel 1 results:


,B,Baseline (ms),Fused (ms),CUDA (ms),Total Speedup,Kernel Speedup,BW (GB/s),Util %
0,1,0.4825,0.3334,0.0168,28.80,19.90,251.3,16.2
1,8,0.4522,0.3607,0.0392,11.52,9.19,858.5,55.2
2,32,0.5383,0.3220,0.1612,3.34,2.00,835.8,53.7
3,64,0.9624,0.6060,0.3130,3.07,1.94,861.0,55.4
